In [5]:
import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter
import os
import glob

In [6]:
#extract data from txt files
def extract_data(file):
    with open(file, 'r', encoding='utf-8') as f:
        return f.read().splitlines()
train_text = extract_data('train_text.txt')
val_text = extract_data('val_text.txt')
test_text = extract_data('test_text.txt')

train_labels = extract_data('train_labels.txt')
val_labels = extract_data('val_labels.txt')
test_labels = extract_data('test_labels.txt')
    

In [7]:
#loads adjective and frequent word lexicons
lex_adj = pd.read_csv("socialsent_hist_adj/adjectives/2000.tsv", sep="\t", names = ["word", "score"])
lex_freq = pd.read_csv("socialsent_hist_freq/frequent_words/2000.tsv", sep="\t", names = ["word", "score"])

In [11]:
#Loads subreddit lexicon, ensuring no duplicates based on first character of filename
subr= glob.glob("socialsent_subreddits/subreddits/*.tsv")
print(subr)
subr_lex = []
seen_chars = set()
for file in subr:
    f_name = os.path.basename(file)
    f_char = f_name[0]
    if f_char.isdigit() and f_char not in seen_chars:
        df = pd.read_csv(file, sep="\t", names = ["word", "score"])
        subr_lex.append(df)
        seen_chars.add(f_char)
    elif f_char.isalpha() and f_char not in seen_chars:
        df = pd.read_csv(file, sep="\t", names = ["word", "score"])
        subr_lex.append(df)
        seen_chars.add(f_char)
subr_lex = pd.concat(subr_lex, ignore_index=True)
print(subr_lex)

['socialsent_subreddits/subreddits\\2007scape.tsv', 'socialsent_subreddits/subreddits\\3DS.tsv', 'socialsent_subreddits/subreddits\\4chan.tsv', 'socialsent_subreddits/subreddits\\ACTrade.tsv', 'socialsent_subreddits/subreddits\\AdviceAnimals.tsv', 'socialsent_subreddits/subreddits\\amiugly.tsv', 'socialsent_subreddits/subreddits\\Anarcho_Capitalism.tsv', 'socialsent_subreddits/subreddits\\Android.tsv', 'socialsent_subreddits/subreddits\\anime.tsv', 'socialsent_subreddits/subreddits\\apple.tsv', 'socialsent_subreddits/subreddits\\archeage.tsv', 'socialsent_subreddits/subreddits\\AskMen.tsv', 'socialsent_subreddits/subreddits\\askscience.tsv', 'socialsent_subreddits/subreddits\\AskWomen.tsv', 'socialsent_subreddits/subreddits\\asoiaf.tsv', 'socialsent_subreddits/subreddits\\atheism.tsv', 'socialsent_subreddits/subreddits\\australia.tsv', 'socialsent_subreddits/subreddits\\aww.tsv', 'socialsent_subreddits/subreddits\\BabyBumps.tsv', 'socialsent_subreddits/subreddits\\baseball.tsv', 'socia

In [ ]:
#create dictionaries for lexicons
adj_dict = dict(zip(lex_adj['word'], lex_adj['score']))
freq_dict = dict(zip(lex_freq['word'], lex_freq['score']))
subr_dict = dict(zip(subr_lex['word'], subr_lex['score']))

In [ ]:
#tokenization function
def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text.split()


In [14]:
#feature extraction function
def extract_features(text):
    words = tokenize(text)
    lex_adj_score =[]
    lex_freq_score = []
    lex_subr_score = []
    for w in words:
        if w in adj_dict:
            lex_adj_score.append(adj_dict[w])
        if w in freq_dict:
            lex_freq_score.append(freq_dict[w])
        if w in subr_dict:
            lex_subr_score.append(subr_dict[w]) 
        #mean sentiment scores for each lexicon
        f1 = np.mean(lex_adj_score) if lex_adj_score else 0
        f2 = np.mean(lex_freq_score) if lex_freq_score else 0
        f3 = np.mean(lex_subr_score) if lex_subr_score else 0

        #lexicon score counts
        f4 = len([x for x in lex_adj_score if x > 0])
        f5 = len([x for x in lex_adj_score if x < 0])

        f6 = len([x for x in lex_freq_score if x > 0])
        f7 = len([x for x in lex_freq_score if x < 0])

        f8 = len([x for x in lex_subr_score if x > 0])
        f9 = len([x for x in lex_subr_score if x < 0])

        #text statistics
        word_count = len(words)
        longest_word = max([len(w) for w in words]) if words else 1
        long_words = len([w for w in words if len(w) >= 5])
        f10 = np.log(word_count + 1)
        f11 = np.log(longest_word + 1)
        f12 = np.log(long_words + 1)
    return [f1, f2, f3, f4, f5, f6, f7, f8, f9, f10, f11, f12]






In [ ]:
#extract features for all datasets
xtrain = np.array([extract_features(text) for text in train_text])
xval = np.array([extract_features(text) for text in val_text])
xtest = np.array([extract_features(text) for text in test_text])

ytrain = np.array(train_labels).astype(int)
yval = np.array(val_labels).astype(int)
ytest = np.array(test_labels).astype(int)

In [16]:
#dataset class
class FeatureDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [17]:
#data loaders
train_loader = DataLoader(FeatureDataset(xtrain, ytrain), batch_size=32, shuffle=True)
val_loader = DataLoader(FeatureDataset(xval, yval), batch_size=32)
test_loader = DataLoader(FeatureDataset(xtest, ytest), batch_size=32)

In [ ]:
#model definition
class LogisticRegression(nn.Module):
    def __init__(self, input_dim, num_classes = 3):
        super(LogisticRegression, self).__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.linear(x)

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LogisticRegression(input_dim=xtrain.shape[1], num_classes=3).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [21]:
def train_model(model, loader):
    model.train()
    total_loss = 0
    for features, labels in loader:
        features, labels = features.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(features)

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)


In [22]:
def evaluate_model(model, loader):
    model.eval()
    preds_list, labels_list = [], []

    with torch.no_grad():
        for features, labels in loader:
            features, labels = features.to(device), labels.to(device)
            logits = model(features)
            preds = torch.argmax(logits, dim=1)

            preds_list.extend(preds.cpu().numpy())
            labels_list.extend(labels.cpu().numpy())
    acc = accuracy_score(labels_list, preds_list)
    f1 = f1_score(labels_list, preds_list, average='weighted')
    return acc, f1

In [23]:
epochs = 10

for e in range(epochs):
    loss = train_model(model, train_loader)
    val_acc, val_f1 = evaluate_model(model, val_loader)

    print(f"Epoch {e+1}: Loss: {loss:.4f}, Validation Accuracy: {val_acc:.4f}, Validation F1-Score: {val_f1:.4f}")

Epoch 1: Loss: 1.0208, Validation Accuracy: 0.4340, Validation F1-Score: 0.2823
Epoch 2: Loss: 1.0176, Validation Accuracy: 0.4345, Validation F1-Score: 0.2685
Epoch 3: Loss: 1.0169, Validation Accuracy: 0.4340, Validation F1-Score: 0.2630
Epoch 4: Loss: 1.0164, Validation Accuracy: 0.4340, Validation F1-Score: 0.2630
Epoch 5: Loss: 1.0156, Validation Accuracy: 0.4360, Validation F1-Score: 0.2742
Epoch 6: Loss: 1.0153, Validation Accuracy: 0.4365, Validation F1-Score: 0.3151
Epoch 7: Loss: 1.0150, Validation Accuracy: 0.4400, Validation F1-Score: 0.2983
Epoch 8: Loss: 1.0147, Validation Accuracy: 0.4360, Validation F1-Score: 0.2744
Epoch 9: Loss: 1.0145, Validation Accuracy: 0.4375, Validation F1-Score: 0.2835
Epoch 10: Loss: 1.0145, Validation Accuracy: 0.4360, Validation F1-Score: 0.2837


In [25]:
test_acc, test_f1 = evaluate_model(model, test_loader)
print("Test Results:")
print(f"Test Accuracy: {test_acc:.4f} \nTest F1-Score: {test_f1:.4f}")

Test Results:
Test Accuracy: 0.4702 
Test F1-Score: 0.3289
